In [ ]:
"""
═══════════════════════════════════════════════════════════════════════════════
  SLEEP APNEA SEVERITY ANALYSIS — LATENT SPACE DEMO
  Biomarker Alignment (Training only — no patient)
═══════════════════════════════════════════════════════════════════════════════

  Context:
    • Biomarkers: T90, SDNN, LF/HF, PWA drops area,
                  δ/θ EEG ratio, α/β EEG ratio, apnea duration
    • Alignment = Pearson r² between biomarker vector and severity score vector
    • Latent space Z : 128-D Gaussian (synthetic, for figure testing)

  Requirements:
    pip install numpy scipy scikit-learn matplotlib seaborn
═══════════════════════════════════════════════════════════════════════════════
"""

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from scipy import stats
from sklearn.manifold import TSNE
from sklearn.linear_model import LinearRegression, HuberRegressor
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
#  GLOBAL STYLE
# ─────────────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":        "DejaVu Sans",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.labelsize":     11,
    "axes.titlesize":     13,
    "axes.titleweight":   "bold",
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "legend.fontsize":    9,
    "figure.facecolor":   "#0e1117",
    "axes.facecolor":     "#161b22",
    "axes.edgecolor":     "#30363d",
    "axes.labelcolor":    "#c9d1d9",
    "xtick.color":        "#8b949e",
    "ytick.color":        "#8b949e",
    "text.color":         "#c9d1d9",
    "grid.color":         "#21262d",
    "grid.linestyle":     "--",
    "grid.linewidth":     0.5,
    "legend.facecolor":   "#161b22",
    "legend.edgecolor":   "#30363d",
    "figure.dpi":         120,
})

SEED = 42
rng  = np.random.default_rng(SEED)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  1. SYNTHETIC DATA GENERATION
#     • Z_train  : (N_TRAIN, 128)  — latent vectors, Gaussian distribution
#     • severity_train : (N_TRAIN,) — severity scores in [0, 1]
#     • bm_train  : dict of (N_TRAIN,) arrays, one per biomarker
#
#  Design choices:
#     – severity is a noisy linear combination of a few latent dimensions
#       so that biomarkers (also correlated with severity) have realistic r²
#     – each biomarker is a noisy projection of Z plus Gaussian noise,
#       with varying correlation strength and sign
# ─────────────────────────────────────────────────────────────────────────────

N_TRAIN = 500
N_LATENT = 128

# ── 128-D Gaussian latent space ──────────────────────────────────────────────
# Mean = 0, isotropic covariance (std ~ 1) — typical VAE posterior
Z_train = rng.standard_normal((N_TRAIN, N_LATENT)).astype(np.float32)

# ── Severity scores: noisy projection of the first few latent dims ───────────
# We use dims 0-4 as the "severity-relevant" subspace
sev_weights = np.zeros(N_LATENT)
sev_weights[:5] = [0.6, 0.4, -0.3, 0.2, 0.1]      # sparse ground truth
raw_severity = Z_train @ sev_weights
# Sigmoid-squash to [0, 1] + small noise
noise_sev = rng.standard_normal(N_TRAIN) * 0.05
severity_train = 1.0 / (1.0 + np.exp(-(raw_severity + noise_sev)))
severity_train = severity_train.astype(np.float32)

# ── Biomarker definitions ────────────────────────────────────────────────────
BIOMARKERS = {
    "T90":            {"unit": "%",    "sev_corr":  0.65, "noise": 0.30},
    "SDNN":           {"unit": "ms",   "sev_corr": -0.55, "noise": 0.35},
    "LF/HF":          {"unit": "au",   "sev_corr":  0.40, "noise": 0.45},
    "PWA drops area": {"unit": "mmHg·s","sev_corr": 0.50, "noise": 0.40},
    "δ/θ EEG ratio":  {"unit": "au",   "sev_corr":  0.35, "noise": 0.50},
    "α/β EEG ratio":  {"unit": "au",   "sev_corr": -0.30, "noise": 0.55},
    "Apnea duration": {"unit": "s",    "sev_corr":  0.70, "noise": 0.25},
}
BM_NAMES = list(BIOMARKERS.keys())
N_BM = len(BM_NAMES)

# Generate each biomarker as:  bm = sev_corr * severity + noise
# (scaled to plausible physiological ranges)
bm_train = {}
for name, cfg in BIOMARKERS.items():
    corr  = cfg["sev_corr"]
    noise = cfg["noise"]
    bm_raw = corr * severity_train + noise * rng.standard_normal(N_TRAIN)
    # Standardize to zero-mean unit-variance for alignment computation
    bm_train[name] = ((bm_raw - bm_raw.mean()) / (bm_raw.std() + 1e-8)).astype(np.float32)

print(f"Z_train shape    : {Z_train.shape}  (mean={Z_train.mean():.3f}, std={Z_train.std():.3f})")
print(f"severity_train   : min={severity_train.min():.3f}  max={severity_train.max():.3f}  "
      f"mean={severity_train.mean():.3f}")
print(f"Biomarkers       : {BM_NAMES}")
print("Synthetic data ready!")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  2. ALIGNMENT COMPUTATION (Pearson r²)
# ═════════════════════════════════════════════════════════════════════════════
print("▶  Computing biomarker alignments …")

def pearson_r2(x: np.ndarray, y: np.ndarray) -> float:
    r, _ = stats.pearsonr(x.ravel(), y.ravel())
    return float(r ** 2)

def signed_r(x: np.ndarray, y: np.ndarray) -> float:
    r, _ = stats.pearsonr(x.ravel(), y.ravel())
    return float(r)

alignment_train = {}

for name in BM_NAMES:
    r2_tr = pearson_r2(bm_train[name], severity_train)
    sr_tr = signed_r(bm_train[name],   severity_train)
    # Keep sign information
    alignment_train[name] = np.sign(sr_tr) * r2_tr

# Sort biomarkers by |alignment| descending for importance ranking
importance_rank = sorted(BM_NAMES, key=lambda n: abs(alignment_train[n]), reverse=True)

print("Alignments computed:")
for n in importance_rank:
    print(f"  {n:<22} signed r² = {alignment_train[n]:+.3f}")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  3. REGRESSION OF SEVERITY VECTOR IN LATENT SPACE
# ═════════════════════════════════════════════════════════════════════════════
print("▶  Regressing severity direction in latent space …")

from sklearn.decomposition import PCA

# Fit linear regression: Z → severity
reg = HuberRegressor(max_iter=500)
reg.fit(Z_train, severity_train)
severity_direction_128 = reg.coef_                     # 128-D vector

print(f"Severity direction norm: {np.linalg.norm(severity_direction_128):.4f}")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  4. FIGURE 1 — SUPERVISED PLS LATENT SPACE (training only)
# ═════════════════════════════════════════════════════════════════════════════
print("▶  Plotting Figure 1 — PLS-supervised latent space …")

from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge

# ── 1. Fit PLS on training data (2 latent components) ──────────────────────
scaler_pls = StandardScaler()
Z_train_sc = scaler_pls.fit_transform(Z_train)

pls = PLSRegression(n_components=2, scale=False, max_iter=1000)
pls.fit(Z_train_sc, severity_train)

# Project training into the 2 PLS components
T_train = pls.transform(Z_train_sc)          # (N_TRAIN, 2)

# ── 2. Variance explained (pseudo-R² of each PLS component vs severity) ────
r2_c1 = stats.pearsonr(T_train[:, 0], severity_train)[0] ** 2
r2_c2 = stats.pearsonr(T_train[:, 1], severity_train)[0] ** 2

# ── 3. Severity iso-contours in PLS space ───────────────────────────────────
reg2d = Ridge(alpha=1e-3)
reg2d.fit(T_train, severity_train)
sev_pred_train = reg2d.predict(T_train)

# Grid for background gradient
t1_min, t1_max = T_train[:, 0].min() - 0.5, T_train[:, 0].max() + 0.5
t2_min, t2_max = T_train[:, 1].min() - 0.5, T_train[:, 1].max() + 0.5
t1_grid, t2_grid = np.meshgrid(
    np.linspace(t1_min, t1_max, 300),
    np.linspace(t2_min, t2_max, 300),
)
grid_pts = np.c_[t1_grid.ravel(), t2_grid.ravel()]
sev_grid = reg2d.predict(grid_pts).reshape(t1_grid.shape)
sev_grid = np.clip(sev_grid, 0, 1)

# ── 4. Severity gradient arrow direction ────────────────────────────────────
sev_arrow = reg2d.coef_ / (np.linalg.norm(reg2d.coef_) + 1e-8)

# ── 5. Plot ─────────────────────────────────────────────────────────────────
fig1 = plt.figure(figsize=(14, 10))
fig1.patch.set_facecolor("#0e1117")

ax_main = fig1.add_subplot(111)
fig1.subplots_adjust(left=0.06, right=0.97, top=0.88, bottom=0.12)
ax_main.set_facecolor("#0d1117")

# Background gradient
ax_main.imshow(
    sev_grid,
    extent=[t1_min, t1_max, t2_min, t2_max],
    origin="lower", aspect="auto",
    cmap="magma", alpha=0.28, vmin=0, vmax=1, zorder=0,
)

# Iso-contours
cs = ax_main.contour(
    t1_grid, t2_grid, sev_grid,
    levels=np.linspace(0.1, 0.9, 9),
    cmap="magma", alpha=0.35, linewidths=0.7, zorder=1,
)
ax_main.clabel(cs, fmt="%.1f", fontsize=7, colors="#8b949e", inline=True)

# Training scatter
sc_tr = ax_main.scatter(
    T_train[:, 0], T_train[:, 1],
    c=severity_train, cmap="YlOrBr",
    s=40, alpha=0.75, linewidths=0.5, rasterized=False,
    norm=Normalize(0, 1), zorder=2,
    label=f"Training apneas (n={N_TRAIN})",
)

# Severity gradient arrow
cx = T_train[:, 0].mean()
cy = T_train[:, 1].mean()
span = max(t1_max - t1_min, t2_max - t2_min) * 0.18
ax_main.annotate(
    "",
    xy=(cx + span * sev_arrow[0], cy + span * sev_arrow[1]),
    xytext=(cx, cy),
    arrowprops=dict(arrowstyle="-|>", color="#ff4444", lw=4.8, mutation_scale=20),
    zorder=6,
)
ax_main.text(
    cx + span * sev_arrow[0] * 1.42,
    cy + span * sev_arrow[1] * 1.12,
    "Severity\ngradient",
    color="#ff4444", fontsize=14, ha="center", va="center",
    fontweight="bold", zorder=6,
)

# Colorbar
cbar_tr = fig1.colorbar(
    ScalarMappable(norm=Normalize(0, 1), cmap="YlOrBr"),
    ax=ax_main, fraction=0.024, pad=0.01, shrink=0.85,
)
cbar_tr.set_label("Severity — Training", color="#e8c47e", labelpad=5)
plt.setp(cbar_tr.ax.yaxis.get_ticklabels(), color="#e8c47e")

# Labels
ax_main.set_xlabel(f"PLS Component 1  (r² w/ severity = {r2_c1:.3f})", fontsize=10)
ax_main.set_ylabel(f"PLS Component 2  (r² w/ severity = {r2_c2:.3f})", fontsize=10)
ax_main.set_title(
    "Supervised PLS Projection  ·  128 → 2 dimensions\n"
    "Axes maximise covariance with severity score  —  background = predicted severity field",
    pad=10, loc="left", fontsize=11,
)

leg_el = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#d4a850",
           markersize=7, label=f"Training apnea (n={N_TRAIN})", linestyle="None"),
    Line2D([0], [0], color="#ff4444", lw=2, label="Regressed severity direction"),
    mpatches.Patch(facecolor="#7a3080", alpha=0.4, label="Severity iso-contours"),
]
ax_main.legend(handles=leg_el, loc="upper left", framealpha=0.3, borderpad=0.8)
ax_main.grid(True, alpha=0.10)

plt.suptitle(
    "Latent Space — Severity-Supervised Projection  (PLS)",
    fontsize=13, fontweight="bold", y=0.97, color="#c9d1d9",
)

fig1.savefig("fig1_pls_latent_space.png", dpi=150, bbox_inches="tight",
             facecolor=fig1.get_facecolor())
print("   ✔  Saved fig1_pls_latent_space.png")
plt.show()

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  5. t-SNE EMBEDDING (training only)
# ═════════════════════════════════════════════════════════════════════════════
print("▶  Running t-SNE (may take ~30s on N=500) …")

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=SEED,
    n_jobs=-1,
)
Z_2d_train = tsne.fit_transform(Z_train)

# Project severity regression vector into 2D via PCA
pca2 = PCA(n_components=2, random_state=SEED)
pca2.fit(Z_train)
sev_dir_2d = pca2.transform(severity_direction_128.reshape(1, -1))[0]
sev_dir_2d /= np.linalg.norm(sev_dir_2d)

print(f"t-SNE done. Z_2d_train shape: {Z_2d_train.shape}")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  6. FIGURE 2 — t-SNE LATENT SPACE (training only)
# ═════════════════════════════════════════════════════════════════════════════
print("▶  Plotting Figure 2 — t-SNE Latent Space …")

fig2, ax = plt.subplots(figsize=(12, 9))
fig2.patch.set_facecolor("#0e1117")
ax.set_facecolor("#0d1117")

# Training scatter
norm_tr = Normalize(vmin=0, vmax=1)
sc_tr = ax.scatter(
    Z_2d_train[:, 0], Z_2d_train[:, 1],
    c=severity_train, cmap="YlOrBr", norm=norm_tr,
    s=45, alpha=0.75, linewidths=0.4, rasterized=False,
    label=f"Training apneas (n={N_TRAIN})",
    zorder=2,
)

# Colorbar
cbar_tr = fig2.colorbar(
    ScalarMappable(norm=norm_tr, cmap="YlOrBr"),
    ax=ax, fraction=0.025, pad=0.01,
)
cbar_tr.set_label("Severity score — Training", color="#e8c47e", labelpad=6)
cbar_tr.ax.yaxis.set_tick_params(color="#e8c47e")
plt.setp(cbar_tr.ax.yaxis.get_ticklabels(), color="#e8c47e")

ax.set_title(
    "Latent Space t-SNE Projection  ·  128 → 2 dimensions\n"
    "Color encodes per-apnea severity score  [0 = benign  →  1 = critical]",
    pad=14, loc="left",
)
ax.set_xlabel("t-SNE dimension 1")
ax.set_ylabel("t-SNE dimension 2")

legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#d4a850",
           markersize=7, label=f"Training apnea (n={N_TRAIN})", linestyle="None"),
]
ax.legend(handles=legend_elements, loc="upper left", framealpha=0.3, borderpad=0.8)
ax.grid(True, alpha=0.15)
plt.tight_layout()

fig2.savefig("fig2_tsne_latent_space.png", dpi=150, bbox_inches="tight",
             facecolor=fig2.get_facecolor())
print("   ✔  Saved fig2_tsne_latent_space.png")
plt.show()

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  7. FIGURE 3 — BIOMARKER ALIGNMENT PLOTS (training only)
# ═════════════════════════════════════════════════════════════════════════════
print("▶  Plotting Figure 3 — Biomarker alignment panels …")

COLS = 4
ROWS = int(np.ceil(N_BM / COLS))
fig3, axes = plt.subplots(ROWS, COLS, figsize=(COLS * 4.8, ROWS * 4.2))
fig3.patch.set_facecolor("#0e1117")
axes_flat = axes.flatten()

def fit_line(x, y):
    m, b, r, p, se = stats.linregress(x, y)
    xr = np.linspace(x.min(), x.max(), 200)
    return xr, m * xr + b, r, p

def pval_str(p: float) -> str:
    if p < 0.001: return "p < 0.001"
    if p < 0.01:  return "p < 0.01"
    if p < 0.05:  return "p < 0.05"
    return f"p = {p:.3f}"

for i, name in enumerate(BM_NAMES):
    ax = axes_flat[i]
    ax.set_facecolor("#0d1117")

    sev_tr = severity_train
    bm_tr  = bm_train[name]

    # Training scatter
    ax.scatter(sev_tr, bm_tr,
               c=sev_tr, cmap="YlOrBr", s=5, alpha=0.4,
               linewidths=0, rasterized=True, zorder=1)

    # Regression line
    xr_tr, yr_tr, r_tr, p_tr = fit_line(sev_tr, bm_tr)
    ax.plot(xr_tr, yr_tr, color="#e8a84a", lw=2.0,
            alpha=0.9, ls="--", label=f"Train  r={r_tr:.2f}", zorder=4)

    ax.set_title(name, fontsize=11, fontweight="bold", pad=6)
    ax.set_xlabel("Severity score", fontsize=9)
    ax.set_ylabel(f"{name}  [{BIOMARKERS[name]['unit']}]", fontsize=9)

    info = (f"Train:   r² = {abs(alignment_train[name]):.3f}  |  {pval_str(p_tr)}")
    ax.text(0.03, 0.97, info,
            transform=ax.transAxes, fontsize=8.5,
            va="top", ha="left", color="#c9d1d9",
            bbox=dict(boxstyle="round,pad=0.35", fc="#1c2128", ec="#30363d", alpha=0.85))

    signed = alignment_train[name]
    sign_sym = "▲" if signed > 0 else "▼"
    sign_col = "#56de91" if signed > 0 else "#fc6e6e"
    ax.text(0.97, 0.15, f"{sign_sym} r² = {abs(signed):.3f}",
            transform=ax.transAxes, fontsize=9, fontweight="bold",
            va="bottom", ha="right", color=sign_col)

    ax.legend(loc="lower right", fontsize=7.5, framealpha=0.3)
    ax.grid(True, alpha=0.12)

# Hide unused axes
for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

fig3.suptitle(
    "Biomarker–Severity Alignment  |  Training set\n"
    "Signed r²: ▲ positive correlation  /  ▼ negative correlation with severity",
    y=1.01, fontsize=12, fontweight="bold",
)
plt.tight_layout()
fig3.savefig("fig3_biomarker_alignment.png", dpi=150, bbox_inches="tight",
             facecolor=fig3.get_facecolor())
print("   ✔  Saved fig3_biomarker_alignment.png")
plt.show()

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  8. FIGURE 4 — BIOMARKER IMPORTANCE RANKING (|r²| training)
# ═════════════════════════════════════════════════════════════════════════════
print("▶  Plotting Figure 4 — Importance ranking …")

fig4, ax4 = plt.subplots(figsize=(11, 5))
fig4.patch.set_facecolor("#0e1117")
ax4.set_facecolor("#0d1117")

vals   = np.array([alignment_train[n] for n in importance_rank])
colors = ["#56de91" if v > 0 else "#fc6e6e" for v in vals]

bars = ax4.barh(importance_rank, vals, color=colors, height=0.55,
                edgecolor="#21262d", linewidth=0.6)

for bar, val in zip(bars, vals):
    sign = "▲ " if val > 0 else "▼ "
    ax4.text(
        val + np.sign(val) * 0.002, bar.get_y() + bar.get_height() / 2,
        f"{sign}{abs(val):.3f}",
        va="center", ha="left" if val > 0 else "right",
        fontsize=9, color="#c9d1d9",
    )

ax4.axvline(0, color="#8b949e", lw=1.2, ls="-")
ax4.set_xlabel("Signed r²  (negative = anti-correlated with severity)", fontsize=10)
ax4.set_title(
    "Biomarker Importance Ranking  —  Alignment with Severity (Training)",
    fontweight="bold", pad=10, loc="left",
)
ax4.grid(True, axis="x", alpha=0.15)

p1 = mpatches.Patch(color="#56de91", label="Positively correlated with severity")
p2 = mpatches.Patch(color="#fc6e6e", label="Negatively correlated with severity")
ax4.legend(handles=[p1, p2], loc="lower right", framealpha=0.3)

plt.tight_layout()
fig4.savefig("fig4_importance_ranking.png", dpi=150, bbox_inches="tight",
             facecolor=fig4.get_facecolor())
print("   ✔  Saved fig4_importance_ranking.png")
plt.show()

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  9. FIGURE 5 — ALIGNMENT SUMMARY HEATMAP (training only)
# ═════════════════════════════════════════════════════════════════════════════
print("▶  Plotting Figure 5 — Alignment summary heatmap …")

fig5, ax5 = plt.subplots(figsize=(9, 3))
fig5.patch.set_facecolor("#0e1117")
ax5.set_facecolor("#0d1117")

summary = np.array([[alignment_train[n] for n in BM_NAMES]])

im = ax5.imshow(summary, cmap="RdYlGn", aspect="auto", vmin=-1.0, vmax=1.0)

ax5.set_xticks(range(N_BM))
ax5.set_xticklabels(BM_NAMES, rotation=30, ha="right", fontsize=9.5)
ax5.set_yticks([0])
ax5.set_yticklabels(["Train r² (signed)"], fontsize=10, fontweight="bold")

for col in range(N_BM):
    val  = summary[0, col]
    sign = "▲" if val > 0 else "▼"
    ax5.text(col, 0, f"{sign}{abs(val):.2f}",
             ha="center", va="center",
             fontsize=8.5, color="black" if abs(val) < 0.6 else "white",
             fontweight="bold")

cbar5 = fig5.colorbar(im, ax=ax5, fraction=0.03, pad=0.02)
cbar5.set_label("Signed r²  (− = anti-aligned,  + = aligned)", fontsize=9)

ax5.set_title(
    "Signed Alignment Summary  ·  r²  ×  sign(r)",
    fontweight="bold", pad=10, loc="left",
)
ax5.spines[:].set_visible(False)

plt.tight_layout()
fig5.savefig("fig5_alignment_heatmap.png", dpi=150, bbox_inches="tight",
             facecolor=fig5.get_facecolor())
print("   ✔  Saved fig5_alignment_heatmap.png")
plt.show()

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  10. CONSOLE SUMMARY REPORT
# ═════════════════════════════════════════════════════════════════════════════
print()
print("═" * 65)
print("  BIOMARKER IMPORTANCE REPORT  —  Training Alignment")
print("═" * 65)
print(f"  {'Biomarker':<22}  {'Signed r²':>10}  {'Direction':>12}")
print("─" * 65)
for n in importance_rank:
    val = alignment_train[n]
    direction = "↑ POSITIVE" if val > 0.05 else "↓ NEGATIVE" if val < -0.05 else "≈ NEUTRAL"
    print(f"  {n:<22}  {val:>+10.3f}  {direction:>12}")
print("═" * 65)
print()
print("  INTERPRETATION:")
print("  • Positive r² → biomarker increases with severity")
print("  • Negative r² → biomarker decreases with severity")
print()
top3 = importance_rank[:3]
print(f"  ⭑ TOP-3 drivers: {', '.join(top3)}")
print()
print("  Figures saved:")
for f in ["fig1_pls_latent_space.png", "fig2_tsne_latent_space.png",
          "fig3_biomarker_alignment.png", "fig4_importance_ranking.png",
          "fig5_alignment_heatmap.png"]:
    print(f"    • {f}")
print("═" * 65)